In [ ]:
import os

import chemiscope
import ipi

import nqetools as nqe
# This follows:
# https://atomistic-cookbook.org/examples/pi-metad/pi-metad.html

In [ ]:
# Make a directory to store everything
directory_min = "min"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"
n_beads = 4
timestep = 1.0  # fs
total_steps = 5000
total_steps_md= 100
stride = 10
temperature = 298
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'zundel'
plumed_type = "mtd-coord"

n_bins = 100
cv_limits = [[0.21, 0.31], [-1, 1]]


In [ ]:
atoms = nqe.read_ipi_xyz("h5o2+.xyz")[-1]
atoms.center()

In [ ]:
# Run minimization
atoms, output_data, output_desc = nqe.run_optimise(directory_min,
                                                   atoms,
                                                   driver=driver_code)

In [ ]:
# Plot the energy of the minimization
nqe.plot_step_energy(output_data)

In [ ]:
# Run unbiased MD
atoms, output_data, output_desc = nqe.run_md(directory_md,
                                             atoms,
                                             driver=driver_code,
                                             total_steps=total_steps_md,
                                             temperature=temperature,
                                             timestep=timestep,
                                             thermostat=thermostat,
                                             md_type=md_type,
                                             stride=stride,
                                             n_beads=1)

In [ ]:
# Plot the energy conservation
nqe.plot_energy_conservation(output_data)

In [ ]:
# Run metadynamics
atoms, output_data, output_desc = nqe.run_plumed_md(directory_meta_md,
                                                    atoms,
                                                    driver=driver_code,
                                                    total_steps=total_steps,
                                                    temperature=temperature,
                                                    timestep=timestep,
                                                    thermostat=thermostat,
                                                    md_type=md_type,
                                                    stride=stride,
                                                    n_beads=1,
                                                    plumed_type=plumed_type)

In [ ]:
colvar_data = ipi.read_trajectory(os.path.join(directory_meta_md, "md.colvar_0"), format="extras")[
    "d,c1.lessthan,c2.lessthan,dc,mtd.bias"
]
traj_data = ipi.read_trajectory(os.path.join(directory_meta_md, "md.pos_0.xyz"))
# Chemiscope plot
chemiscope.show(
    frames=traj_data,
    properties=dict(
        d_OO=10 * colvar_data[:, 0],  # nm to Å
        delta_coord=colvar_data[:, 1],
        bias=27.211386 * output_data["ensemble_bias"],  # Ha to eV
        time=2.4188843e-05 * output_data["time"],  # atomictime to ps
    ),  # attime to ps
    settings=chemiscope.quick_settings(
        x="d_OO", y="delta_coord", z="bias", color="time", trajectory=True
    ),
    mode="default",
)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data)

In [ ]:
# Plot the energy conservation
nqe.plot_energy_conservation(output_data)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data)

In [ ]:
# Run the hills command
nqe.run_plumed_hills(directory_meta_md, bins=n_bins, cv=cv_limits)

In [ ]:
# Load the free energy surface data
fes_arrays = nqe.load_fes_data(directory_meta_md, n_bins)
# Plot the free energy surface convergence
nqe.plot_energy_contour_series(fes_arrays)

In [ ]:
# Run PIMD metadynamics
atoms, output_data, output_desc = nqe.run_plumed_md(directory_meta_pimd,
                                                    atoms,
                                                    driver=driver_code,
                                                    total_steps=total_steps,
                                                    temperature=temperature,
                                                    timestep=timestep,
                                                    thermostat=thermostat,
                                                    md_type=md_type,
                                                    stride=stride,
                                                    n_beads=n_beads,
                                                    plumed_type=plumed_type)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data)

In [ ]:
# Plot the energy conservation
nqe.plot_energy_conservation(output_data)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data)

In [ ]:
# Run the hills command
nqe.run_plumed_hills(directory_meta_pimd, bins=n_bins, cv=cv_limits)

In [ ]:
# Load the free energy surface data
fes_arrays = nqe.load_fes_data(directory_meta_pimd, n_bins)
# Plot the free energy surface convergence
nqe.plot_energy_contour_series(fes_arrays)